# Fine-tuning Gemma 4 E2B for Nutritional Analysis

This notebook implements the fine-tuning of the Gemma 4 E2B model using QLoRA on the Codatta/MM-Food-100K dataset for nutritional analysis in the MealTracking Android app.

## 1. Set Up Your Training Framework

Install and configure the Unsloth library combined with Hugging Face's TRL framework for optimized training of Gemma 4 models.

In [ ]:
# Install dependencies
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install transformers datasets accelerate peft trl torch huggingface_hub

In [3]:
# Import libraries
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
import json

d:\Projects\MealTracking\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0421 11:58:00.754000 9828 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


## 2. Format the Codatta Dataset

Load the Codatta/MM-Food-100K dataset and transform it into a conversational message format with user turns (image + instruction) and assistant turns (structured JSON output).

In [4]:
# Login to Hugging Face to access private or gated datasets
!huggingface-cli login


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [ ]:
# Load the Codatta/MM-Food-100K dataset
dataset = load_dataset("Codatta/MM-Food-100K", split="train")

# Note: Since Gemma is text-only, we'll use the text descriptions from the JSON metadata
# For multimodal, you might need a vision model, but here we assume text-based training

def format_conversation(example):
    # Assuming the dataset has 'image' and 'metadata' fields
    # metadata is a dict with nutritional info
    user_message = "Analyze the nutritional content and ingredients of this meal."
    assistant_message = json.dumps(example['metadata'])  # Structured JSON
    
    return {
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
    }

# Format the dataset
formatted_dataset = dataset.map(format_conversation)

Map:  72%|███████▏  | 72000/100000 [00:04<00:01, 17297.68 examples/s]

## 3. Set the QLoRA Hyperparameters

Configure QLoRA settings including learning rate (e.g., 2e-4), LoRA rank (e.g., 16), lora_alpha (e.g., 32), and epochs (e.g., 1-5), then run the fine-tuning training loop.

In [ ]:
# Load the Gemma 4B model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4b-it",  # Gemma 4B instruction-tuned
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Twice the rank
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,  # For 1 epoch, adjust based on dataset size
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
)

# Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",  # Assuming formatted as text
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Train
trainer.train()

## 4. Merge and Export to GGUF

Merge the trained LoRA adapter weights back into the base Gemma 4 model and export the unified model to GGUF format using Unsloth or llama.cpp scripts.

In [ ]:
# Merge LoRA weights back into the base model
model = FastLanguageModel.for_inference(model)  # Unload LoRA for merging
model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")

In [ ]:
# Export to GGUF format
model.save_pretrained_gguf("gemma_finetuned_gguf", tokenizer)